# E8 수치 중심 재현

E6 실제 LB0.6109를 기준으로 수치 입력과 순위 학습을 비교한다. README.md에 실험과 한계를 기록했다. requirements.txt를 설치한 Python3.12 커널과 프로젝트 원본/E1/E6/E7 캐시가 필요하다.


In [ ]:
from pathlib import Path
import json, subprocess, sys
root=Path.cwd().resolve()
for p in [root,*root.parents]:
    if (p/"work_현호/Modeling/E8/run_numeric.py").exists():
        root=p
        break
else:
    raise RuntimeError("프로젝트 폴더 안에서 실행하세요.")
folder=root/"work_현호/Modeling/E8"
print(folder)


## 실행 범위

기본값은 저장 결과 확인이다. 새로 학습하려면 RUN_TRAINING=True로 바꾼다. E8 결과가 덮어써지므로 새 데이터/환경에서는 새 결과 폴더를 사용한다.


In [ ]:
RUN_TRAINING=False
if RUN_TRAINING:
    for name in ["build_numeric.py","run_numeric.py","refine_numeric.py","package_results.py"]:
        subprocess.run([sys.executable,"-u",str(folder/name)],cwd=root,check=True)


In [ ]:
import pandas as pd
decision=json.loads((folder/"decision.json").read_text())
print("E6 실제 LB:",decision["baseline_actual_LB"],"새 LB:",decision["new_actual_LB"])
print("추천 구성:",decision["selected_spec"])
metrics=pd.read_csv(folder/"metrics.csv").set_index("candidate")
print(metrics.loc[list(dict.fromkeys(decision["outputs"].values())),["AP","top20_TP","eligible","reasons"]].to_string())
print(pd.read_csv(folder/"bootstrap.csv").to_string(index=False))


## 재현 검증

저장 모델 재로드와 독립 AP·순위·혼합 계산 확인이다. 예측 성능의 독립 미래 검증을 의미하지 않는다.


In [ ]:
subprocess.run([sys.executable,"-u",str(folder/"run_numeric.py"),"--verify"],cwd=root,check=True)
subprocess.run([sys.executable,str(folder/"verify_arithmetic.py")],cwd=root,check=True)
submission=pd.read_csv(folder/"submission_recommended.csv")
print(submission.head().to_string(index=False))
print("rows",len(submission),"file",folder/"submission_recommended.csv")
